In [1]:
import sys
from pathlib import Path

# Walk up from backend/test/ to the project root
project_root = Path().resolve().parent.parent
sys.path.insert(0, str(project_root))

print(f"Project root: {project_root}") 

Project root: /Users/liusiyi/Desktop/Agent/trip-planner


In [2]:
from backend.graph.weather import weather_subgraph
from backend.graph.attraction import attraction_subgraph
from backend.graph.hotel import hotel_subgraph
from backend.models import DayWeatherInfo

from langchain_core.messages import AIMessage, ToolMessage
from backend.graph.orchestrator import orchestrator
from langchain_core.messages import HumanMessage

from backend.graph.orchestrator import plan_trip

In [3]:
USER_QUERY = "I'm traveling to Chicago from 5 July to 6 July, 2026. Help me plan my trip."

# Sub Graph Testing

## Weather

In [10]:
# ── Weather ──────────────────────────────────────────────
result = await weather_subgraph.ainvoke({"user_query": USER_QUERY})

assert result["weather_info"], "weather_info should not be empty"
assert all(w.date for w in result["weather_info"]), "every DayWeatherInfo should have a date"
print(f"✅ Weather: {len(result['weather_info'])} days")
for w in result["weather_info"]:
    print(f"   {w.date}: {w.condition}, {w.temp_min_c}–{w.temp_max_c}°C")

✅ Weather: 2 days
   2026-06-28: Overcast, 17.0–26.4°C
   2026-06-29: Thunderstorm, 22.8–36.0°C


## Attractions

In [4]:
# ── Attractions ──────────────────────────────────────────
# Test without weather first (the "parallel" case)
result = await attraction_subgraph.ainvoke({
    "user_query": USER_QUERY,
    "weather_info": [],
})
assert result["attractions"], "attractions should not be empty"
print(f"\n✅ Attractions (no weather): {len(result['attractions'])} found")


✅ Attractions (no weather): 30 found


In [5]:
result

{'attractions': [AttractionInfo(name='Daring Bank Heist Adventure', attraction_type='Adventure Tour', description="Embark on a daring bank heist adventure across Chicago's streets.", latitude=None, longitude=None, booking_required=None, estimated_fee_usd=None, fee_notes=None),
  AttractionInfo(name='Wrigley Field', attraction_type='Historical Site', description="Experience the thrill of America's favorite pastime at historic Wrigley Field, home of the Chicago Cubs!", latitude=41.9484384, longitude=-87.6553327, booking_required=None, estimated_fee_usd=None, fee_notes=None),
  AttractionInfo(name='The Guess Who Concert', attraction_type='Concert', description='Get ready to experience a night of classic rock with The Guess Who at the Huntington Bank Pavilion at Northerly Island in Chicago!', latitude=None, longitude=None, booking_required=None, estimated_fee_usd=None, fee_notes=None),
  AttractionInfo(name='Huntington Bank Pavilion at Northerly Island', attraction_type='Concert Venue', de

In [7]:
# Test with weather context (the "sequential" case)
mock_weather = [
    DayWeatherInfo(location="Chicago", date="2026-06-25", condition="Light drizzle",
                   temp_min_c=17.2, temp_max_c=25.9, precipitation_mm=0.4),
]
result = await attraction_subgraph.ainvoke({
    "user_query": USER_QUERY,
    "weather_info": mock_weather,
})
print(f"✅ Attractions (with weather): {len(result['attractions'])} found")
for a in result["attractions"]:
    print(f"   {a.name} ({a.attraction_type}) lat={a.latitude} lon={a.longitude}")

✅ Attractions (with weather): 12 found
   Underwater Safaris (null) lat=41.935458 lon=-87.66293569999999
   Big Mini Putt Club (null) lat=42.9611936 lon=-85.6699131
   Shit Fountain (null) lat=None lon=None
   Trickery (null) lat=41.945174 lon=-87.6492687
   { (null) lat=None lon=None
   The Field Museum (museum) lat=41.866261 lon=-87.6169805
   Adler Planetarium (museum) lat=41.8664066 lon=-87.6067898
   The Art Institute (description) lat=41.8796031 lon=-87.6223504
   Field Museum (Museum) lat=41.866261 lon=-87.6169805
   Fox in a Box Escape Rooms (escape room) lat=-27.4696768 lon=153.0278423
   Fox in a Box Escape Room Chicago (Escape Room) lat=41.8718392 lon=-87.62875009999999
   Art Institute of Chicago (museum) lat=41.8796031 lon=-87.6223504


## Hotel

In [8]:
# ── Hotels ───────────────────────────────────────────────
result = await hotel_subgraph.ainvoke({
    "user_query": USER_QUERY,
    "weather_info": [],
    "attractions": [],
})
assert result["hotels"], "hotels should not be empty"
print(f"\n✅ Hotels: {len(result['hotels'])} found")
for h in result["hotels"]:
    min_price = min((o.price_usd for o in h.offers), default=None)
    print(f"   {h.name} — from ${min_price}/night")


✅ Hotels: 5 found
   LondonHouse Chicago, Curio Collection by Hilton — from $None/night
   Swissôtel Chicago — from $None/night
   Hyatt Regency Chicago — from $None/night
   The Langham, Chicago — from $None/night
   Hilton Chicago — from $None/night


In [4]:
from langchain_core.messages import AIMessage, ToolMessage

def _short(value, max_len=120) -> str:
    s = str(value)
    return s if len(s) <= max_len else s[:max_len] + "…"

async for event in hotel_subgraph.astream_events(
    {
        "user_query": USER_QUERY,
        "weather_info": [],
        "attractions": [],
    },
    version="v2",
):
    kind = event["event"]
    name = event.get("name", "")
    data = event.get("data", {})

    # ── Node boundaries ──────────────────────────────────
    if kind == "on_chain_start" and name in ("hotel_agent", "parse_hotel_results", "tools"):
        print(f"\n{'─'*50}")
        print(f"▶ {name}")

    # ── Tool calls (scheduled by LLM) ────────────────────
    elif kind == "on_chat_model_end":
        output = data.get("output")
        if output and getattr(output, "tool_calls", None):
            for tc in output.tool_calls:
                print(f"\n  [tool_call] {tc['name']}")
                for k, v in tc.get("args", {}).items():
                    print(f"    {k}: {_short(v)}")

    # ── Tool execution ───────────────────────────────────
    elif kind == "on_tool_start":
        args = data.get("input", {})
        print(f"\n  [tool →] {name}")
        for k, v in args.items():
            print(f"    {k}: {_short(v)}")

    elif kind == "on_tool_end":
        print(f"  [tool ←] {name} returned: {data.get('output', '')}")

    # ── AI message tokens ────────────────────────────────
    elif kind == "on_chat_model_stream":
        chunk = data.get("chunk")
        if not chunk:
            continue
        content = chunk.content
        if isinstance(content, str) and content:
            print(content, end="", flush=True)
        elif isinstance(content, list):
            for block in content:
                if isinstance(block, dict) and block.get("type") == "text":
                    print(block.get("text", ""), end="", flush=True)

    # ── Final node outputs ───────────────────────────────
    elif kind == "on_chain_end" and name == "parse_hotel_results":
        output = data.get("output", {})
        hotels = output.get("hotels", [])
        print(f"\n\n  [parse_hotel_results] → {len(hotels)} hotel(s) structured")
        for h in hotels:
            print(f"    {h.name} ({len(h.offers)} offers)")


──────────────────────────────────────────────────
▶ hotel_agent

  [tool_call] search_hotels
    city: Chicago
    chk_out: 2026-06-29
    chk_in: 2026-06-28

──────────────────────────────────────────────────
▶ tools

  [tool →] search_hotels
    city: Chicago
    chk_out: 2026-06-29
    chk_in: 2026-06-28
  [tool ←] search_hotels returned: content='{"city": "Chicago", "chk_in": "2026-06-28", "chk_out": "2026-06-29", "hotels": [{"name": "LondonHouse Chicago, Curio Collection by Hilton", "hotel_key": "g35805-d9145578", "rating": null, "address": null, "offers": []}, {"name": "Swissôtel Chicago", "hotel_key": "g35805-d114581", "rating": null, "address": null, "offers": []}, {"name": "Hyatt Regency Chicago", "hotel_key": "g35805-d87617", "rating": null, "address": null, "offers": []}, {"name": "The Langham, Chicago", "hotel_key": "g35805-d4046139", "rating": null, "address": null, "offers": []}, {"name": "Hilton Chicago", "hotel_key": "g35805-d87590", "rating": null, "address": null, "

In [5]:
from backend.graph.hotel import search_hotels

In [8]:
result = await search_hotels.coroutine("San Francisco", chk_out="2026-06-29", chk_in="2026-06-28")
result

{'city': 'San Francisco',
 'chk_in': '2026-06-28',
 'chk_out': '2026-06-29',
 'hotels': [{'name': 'Hotel Best San Francisco',
   'hotel_key': 'g562814-d500568',
   'rating': None,
   'address': None,
   'offers': []},
  {'name': 'Hotel Zephyr San Francisco',
   'hotel_key': 'g60713-d81222',
   'rating': None,
   'address': None,
   'offers': []},
  {'name': 'Hyatt Regency San Francisco',
   'hotel_key': 'g60713-d81103',
   'rating': None,
   'address': None,
   'offers': []},
  {'name': 'Hotel Fairmont San Francisco',
   'hotel_key': 'g60713-d81397',
   'rating': None,
   'address': None,
   'offers': []},
  {'name': 'Palace Hotel, A Luxury Collection Hotel, San Francisco',
   'hotel_key': 'g60713-d115617',
   'rating': None,
   'address': None,
   'offers': []}]}

# Orchestrator Testing

In [ ]:


final_state = await plan_trip(
    USER_QUERY
)

# Structural checks
assert final_state["weather_info"], "weather_info missing"
assert final_state["attractions"], "attractions missing"
assert final_state["hotels"], "hotels missing"
assert final_state["messages"], "no messages — itinerary agent didn't respond"

# The last message should be the itinerary from the LLM
itinerary_response = final_state["messages"][-1]
print("✅ Orchestrator completed\n")
print("=== ITINERARY ===")
print(itinerary_response.content)

In [4]:
# ── Streaming observer ───────────────────────────────────────────────────────



initial_state = {
    "messages": [HumanMessage(content=USER_QUERY)],
    "user_query": USER_QUERY,
    "weather_info": [], "attractions": [], "hotels": [],
    "errors": [], "execution_plan": [],
}

# Subgraph node names to track (LangGraph prefixes them with their parent node name)
SUBGRAPH_NODES = {
    "weather_agent", "attraction_agent", "hotel_agent",
    "tools", "parse_weather_results", "extract_attractions",
    "geocode_attractions", "parse_hotel_results",
}
ORCHESTRATOR_NODES = {"plan_execution", "weather", "attractions", "hotels", "itinerary"}

def _short(value, max_len=120) -> str:
    s = str(value)
    return s if len(s) <= max_len else s[:max_len] + "…"

print(f"{'='*60}")
print(f"  TRIP PLANNING: {USER_QUERY}")
print(f"{'='*60}\n")

current_agent = None   # track which top-level agent we're inside

async for event in orchestrator.astream_events(initial_state, version="v2"):
    kind  = event["event"]
    name  = event.get("name", "")
    data  = event.get("data", {})
    tags  = event.get("tags", [])

    # ── Orchestrator-level node boundaries ───────────────────────────────────
    if kind == "on_chain_start" and name in ORCHESTRATOR_NODES:
        if name == "plan_execution":
            print("📋 Planning execution order…")
        elif name in ("weather", "attractions", "hotels"):
            current_agent = name
            label = {"weather": "🌤  Weather", "attractions": "🗺  Attractions", "hotels": "🏨  Hotels"}[name]
            print(f"\n{'─'*60}")
            print(f"{label} Agent starting")
            print(f"{'─'*60}")
        elif name == "itinerary":
            print(f"\n{'─'*60}")
            print(f"📅  Itinerary Agent starting")
            print(f"{'─'*60}")

    elif kind == "on_chain_end" and name in ORCHESTRATOR_NODES:
        output = data.get("output", {})
        if name == "plan_execution":
            plan = output.get("execution_plan", [])
            print(f"   Order: {' → '.join(str(s) for s in plan)}\n")
        elif name == "weather":
            days = output.get("weather_info") or []
            print(f"\n   ✅ Weather complete — {len(days)} day(s) retrieved")
        elif name == "attractions":
            places = output.get("attractions") or []
            print(f"\n   ✅ Attractions complete — {len(places)} place(s) found")
        elif name == "hotels":
            hotels = output.get("hotels") or []
            print(f"\n   ✅ Hotels complete — {len(hotels)} hotel(s) found")
        elif name == "itinerary":
            print(f"\n   ✅ Itinerary complete")

    # ── Subgraph node boundaries ─────────────────────────────────────────────
    elif kind == "on_chain_start" and name in SUBGRAPH_NODES:
        labels = {
            "weather_agent":       "  [agent] Weather LLM thinking…",
            "attraction_agent":    "  [agent] Attraction LLM thinking…",
            "hotel_agent":         "  [agent] Hotel LLM thinking…",
            "parse_weather_results":  "  [parse] Parsing weather results…",
            "extract_attractions":    "  [parse] Extracting attractions from search…",
            "geocode_attractions":    "  [parse] Geocoding attractions…",
            "parse_hotel_results":    "  [parse] Parsing hotel results…",
        }
        if name in labels:
            print(f"\n{labels[name]}")

    # ── Tool calls ───────────────────────────────────────────────────────────
    elif kind == "on_tool_start":
        args = data.get("input", {})
        print(f"\n  [tool →] {name}")
        for k, v in args.items():
            print(f"           {k}: {_short(v)}")

    elif kind == "on_tool_end":
        output = data.get("output", "")
        # ToolMessage content is the raw API response — truncate for readability
        print(f"  [tool ←] {name} returned: {_short(output)}")

    # ── AI message streaming (token by token) ────────────────────────────────
    elif kind == "on_chat_model_stream":
        chunk = data.get("chunk")
        if not chunk:
            continue
        content = chunk.content

        # content can be a string (most providers) or a list of blocks (Gemini)
        if isinstance(content, str) and content:
            print(content, end="", flush=True)
        elif isinstance(content, list):
            for block in content:
                if isinstance(block, dict) and block.get("type") == "text":
                    text = block.get("text", "")
                    if text:
                        print(text, end="", flush=True)

    # ── Completed AI messages (full, with tool_calls visible) ────────────────
    elif kind == "on_chat_model_end":
        output = data.get("output")
        if not output:
            continue
        msg = output if isinstance(output, AIMessage) else getattr(output, "generations", [[None]])[0][0].message if hasattr(output, "generations") else None
        if msg and getattr(msg, "tool_calls", None):
            print()  # newline after streamed tokens
            for tc in msg.tool_calls:
                print(f"\n  [tool_call scheduled] {tc['name']}")
                for k, v in tc.get("args", {}).items():
                    print(f"           {k}: {_short(v)}")

  TRIP PLANNING: I'm traveling to Chicago from 5 July to 6 July, 2026. Help me plan my trip.


────────────────────────────────────────────────────────────
🌤  Weather Agent starting
────────────────────────────────────────────────────────────

  [agent] Weather LLM thinking…


  [tool_call scheduled] get_weather_forecast
           city: Chicago
           end_date: 2026-07-06
           start_date: 2026-07-05

  [tool →] get_weather_forecast
           city: Chicago
           end_date: 2026-07-06
           start_date: 2026-07-05
  [tool ←] get_weather_forecast returned: content='{"city": "Chicago", "latitude": 41.85003, "longitude": -87.65005, "forecast": [{"date": "2026-07-05", "temp_max…

  [agent] Weather LLM thinking…
**Chicago – 5 July 2026**  
- Thunderstorm, max 33 °C, min 20 °C, ≈1.8 mm precipitation.  

**Chicago – 6 July 2026**  
- Moderate drizzle, max 27 °C, min 21 °C, ≈3.3 mm precipitation.
  [parse] Parsing weather results…

   ✅ Weather complete — 2 day(s) retrieved



In [5]:
event

{'event': 'on_chain_end',
 'data': {'output': {'messages': [HumanMessage(content="I'm traveling to Chicago from 5 July to 6 July, 2026. Help me plan my trip.", additional_kwargs={}, response_metadata={}, id='37e28871-73bc-4a67-8da6-130b2eb29230')],
   'user_query': "I'm traveling to Chicago from 5 July to 6 July, 2026. Help me plan my trip.",
   'weather_info': [DayWeatherInfo(location='Chicago', date='2026-07-05', temp_min_c=19.6, temp_max_c=33.2, precipitation_mm=1.8, condition='Thunderstorm', error=None),
    DayWeatherInfo(location='Chicago', date='2026-07-06', temp_min_c=20.9, temp_max_c=26.9, precipitation_mm=3.3, condition='Moderate drizzle', error=None)],
   'attractions': [AttractionInfo(name='Reid Murdoch Building', attraction_type='Event Venue', description='Multiple experiences per day. Visit TheateroftheMindChicago.com for dates, times, and tickets.', latitude=41.8880852, longitude=-87.63226110000001, booking_required=True, estimated_fee_usd=None, fee_notes=None),
    Attr

In [5]:
final_state = await plan_trip(USER_QUERY)

itinerary = final_state["itinerary"]
print(f"Destination: {itinerary.destination}")
print(f"Summary: {itinerary.trip_summary}\n")

for day in itinerary.days:
    print(f"── {day.date} ({day.weather_summary}) ──")
    for activity in day.activities:
        print(f"  {activity.time}  {activity.name} ({activity.estimated_duration_hours}h)")
        if activity.notes:
            print(f"           ↳ {activity.notes}")
    if day.hotel:
        print(f"  🏨 Stay: {day.hotel}")
    print()

if itinerary.travel_tips:
    print("Travel tips:")
    for tip in itinerary.travel_tips:
        print(f"  • {tip}")

Destination: Chicago
Summary: This 2-day trip to Chicago includes a mix of indoor and outdoor activities, considering the weather forecast. The first day will focus on indoor attractions due to the thunderstorm, while the second day will include a mix of indoor and outdoor activities.

── 2026-07-05 (Thunderstorm with hail, 21.6–33.6°C) ──
  9:00 AM  The Art Institute of Chicago (2.0h)
  12:00 PM  Chicago Museum of Ice Cream (1.5h)
  2:30 PM  The Chicago Theatre (2.0h)
  🏨 Stay: LondonHouse Chicago, Curio Collection by Hilton

── 2026-07-06 (Light drizzle, 20.7–28.2°C) ──
  9:30 AM  Skydeck Chicago (1.5h)
  11:30 AM  Chicago Underground Pedway & Downtown Secrets Walking Tour (2.0h)
  2:00 PM  Chicago Sports Museum (1.5h)
  🏨 Stay: LondonHouse Chicago, Curio Collection by Hilton

Travel tips:
  • Check the weather forecast before heading out each day
  • Consider purchasing a Chicago CityPASS for discounted attraction tickets


In [6]:
final_state

{'messages': [HumanMessage(content="I'm traveling to Chicago from 5 July to 6 July, 2026. Help me plan my trip.", additional_kwargs={}, response_metadata={}, id='296e75cc-2885-4455-93d4-7130cdf7d98a')],
 'user_query': "I'm traveling to Chicago from 5 July to 6 July, 2026. Help me plan my trip.",
 'weather_info': [DayWeatherInfo(location='Chicago', date='2026-07-05', temp_min_c=21.6, temp_max_c=33.6, precipitation_mm=4.2, condition='Thunderstorm with hail', error=None),
  DayWeatherInfo(location='Chicago', date='2026-07-06', temp_min_c=20.7, temp_max_c=28.2, precipitation_mm=0.6, condition='Light drizzle', error=None)],
 'attractions': [AttractionInfo(name='Skydeck Chicago', attraction_type=None, description=None, latitude=41.87887610000001, longitude=-87.635915, booking_required=None, estimated_fee_usd=None, fee_notes=None),
  AttractionInfo(name='Balloon Museum Chicago', attraction_type=None, description=None, latitude=41.88325, longitude=-87.6323879, booking_required=None, estimated_